# 🔥 WiDS 2026 | CV-Bagged GBSA×LGB Fusion (Proven + 20 seeds)
Test predictions from **fold models averaged** (CV-bag architecture).

**Config**: W24=0.95, W48=0.45, PowerCal24=1.1, Seeds=20, Configs=6, Cutoff=5km
**Total models**: 6 × 20 × 5 + 2 × 25 × 5 = 600 GBSA + 250 LGB


In [1]:
!pip install -q scikit-survival

import numpy as np
import pandas as pd
import warnings
import os
import time as timer
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv

warnings.filterwarnings('ignore')
np.random.seed(777)
HORIZONS_PRED = np.array([12, 24, 48, 72], dtype=float)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 18.8 MB/s eta 0:00:00


In [2]:
def locate_datasets():
    train_path, test_path = 'train.csv', 'test.csv'
    for search_root in ['/kaggle/input', '../input']:
        if os.path.exists(search_root):
            for root, _, files in os.walk(search_root):
                if 'train.csv' in files: train_path = os.path.join(root, 'train.csv')
                if 'test.csv' in files: test_path = os.path.join(root, 'test.csv')
    return train_path, test_path

train_path, test_path = locate_datasets()
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print(f'Training: {len(train_df)} samples, Test: {len(test_df)} samples')

Training: 221 samples, Test: 95 samples


In [3]:
def create_features(df):
    result = df.copy()
    dist = result['dist_min_ci_0_5h'].clip(lower=1)
    speed = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']
    result['log_distance'] = np.log1p(dist)
    result['inv_distance'] = 1 / (dist / 1000 + 0.1)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance'] = np.sqrt(dist)
    result['dist_km'] = dist / 1000
    result['dist_km_sq'] = (dist / 1000) ** 2
    result['dist_rank'] = dist.rank(pct=True)
    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio'] = area_first / (dist / 1000 + 0.1)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)
    result['has_movement'] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result['eta_hours'] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result['log_eta'] = np.log1p(result['eta_hours'].clip(0, 9999))
    radial_growth = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result['threat_score'] = result['alignment_abs'] * speed / np.log1p(dist)
    result['fire_urgency'] = perimeters * speed
    result['growth_intensity'] = result['area_growth_rate_ha_per_h'] * perimeters
    result['zone_critical'] = (dist < 5000).astype(float)
    result['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result['zone_safe'] = (dist >= 10000).astype(float)
    result['is_summer'] = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) & (result['event_start_hour'] < 20)).astype(float)
    drop_cols = ['relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
                 'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h']
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result

train_processed = create_features(train_df)
test_processed = create_features(test_df)

In [4]:
def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds

def sigmoid_pred(dist, threshold, scale):
    return 1.0 / (1.0 + np.exp((dist - threshold) / scale))

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        if at_risk > 0: surv[i] = 1 - censored_at_t / at_risk
        if i > 0: surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side='right') - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0
    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon: weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon: weights[i] = 1.0 / G(horizon)
    return weights

def enforce_monotonicity(preds):
    result = np.clip(preds, 0, 1)
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i-1])
    return result

In [5]:
X_surv_train = train_df.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_surv_test = test_df.drop(columns=['event_id'])
y_surv = Surv.from_arrays(event=train_df['event'].astype(bool), time=train_df['time_to_hit_hours'])
event_values = train_df['event'].values
time_values = train_df['time_to_hit_hours'].values
dist_test = test_df['dist_min_ci_0_5h'].values

gbsa_configs = [
    {'learning_rate': 0.01, 'subsample': 0.7,  'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.01, 'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 15, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.01, 'subsample': 0.6,  'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.005,'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 2000, 'dropout_rate': 0.0},
    {'learning_rate': 0.01, 'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 20, 'min_samples_split': 3, 'n_estimators': 1400, 'dropout_rate': 0.0},
    {'learning_rate': 0.001, 'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 20, 'min_samples_split': 3, 'n_estimators': 2000, 'dropout_rate': 0.0},
]
SEEDS = (123, 456, 789, 777, 666,
         1511, 1523, 2025, 2026, 2033,
         279, 239, 70, 77, 31,
         2024, 2077, 3077, 123456, 654321,
         4640, 841, 7755, 8525, 2701,
         8817, 8864, 4085, 8919, 934,
         4746, 1699, 7401, 7826, 4098,
         2921, 1204, 2752, 8384, 1284)
N_SEEDS = len(SEEDS)

test_gbsa = np.zeros((len(X_surv_test), 4))
total_models = len(gbsa_configs) * N_SEEDS * 5
model_count = 0
t_start = timer.time()

for cfg_idx, cfg in enumerate(gbsa_configs):
    cfg_test = np.zeros((len(X_surv_test), 4))
    for seed in SEEDS:
        seed_test = np.zeros((len(X_surv_test), 4))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X_surv_train, event_values)):
            m = GradientBoostingSurvivalAnalysis(**{**cfg, 'random_state': seed})
            m.fit(X_surv_train.iloc[tr_idx], y_surv[tr_idx])
            seed_test += get_surv_predictions(m, X_surv_test) / 5
            model_count += 1
        cfg_test += seed_test / N_SEEDS
    test_gbsa += cfg_test / len(gbsa_configs)
    elapsed = timer.time() - t_start
    print(f'Config {cfg_idx+1}/{len(gbsa_configs)} done [{model_count}/{total_models}, {elapsed/60:.1f}m]')

# PowerCal 24h
test_gbsa[:, 1] = np.clip(test_gbsa[:, 1] ** 1.1, 0, 1)
print(f'GBSA done: {total_models} fold models')

Config 1/6 done [200/1200, 4.3m]
Config 2/6 done [400/1200, 8.9m]
Config 3/6 done [600/1200, 13.0m]
Config 4/6 done [800/1200, 20.6m]
Config 5/6 done [1000/1200, 25.7m]
Config 6/6 done [1200/1200, 33.0m]
GBSA done: 1200 fold models


In [6]:
X_lgb_train = train_processed.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_lgb_test = test_processed.drop(columns=['event_id'])

lgb_cfgs = {
    24: {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 300,
         'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_samples': 8,
         'reg_alpha': 0.5, 'reg_lambda': 2.0, 'num_leaves': 7},
    48: {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 200,
         'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 5,
         'reg_alpha': 0.1, 'reg_lambda': 1.0, 'num_leaves': 4},
}
LGB_SEEDS = (123, 456, 789, 777, 666,
             1511, 1523, 2025, 2026, 2033,
             279, 239, 70, 77, 31,
             2024, 2077, 3077, 123456, 654321,
             2034, 2035, 2036, 1984, 1991,
             3255, 1011, 6241, 2790, 6847,
             8141, 7752, 432, 906, 6217,
             7785, 1603, 7609, 965, 2506,
             3771, 7080, 4963, 7939, 2751,
             473, 339, 3675, 5535, 4760
            )
N_LGB_SEEDS = len(LGB_SEEDS)
lgb_test = {}

for horizon in [24, 48]:
    y_bin, mask = make_binary_target(time_values, event_values, horizon)
    valid_idx = np.where(mask)[0]
    cfg = lgb_cfgs[horizon]
    all_test = np.zeros(len(X_lgb_test))
    for seed in LGB_SEEDS:
        seed_test = np.zeros(len(X_lgb_test))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_v, va_v in cv.split(valid_idx, y_bin[mask]):
            tr_idx = valid_idx[tr_v]
            ipcw_w = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            m = lgb.LGBMClassifier(**cfg, objective='binary', random_state=seed, verbose=-1)
            m.fit(X_lgb_train.iloc[tr_idx], y_bin[tr_idx], sample_weight=ipcw_w)
            seed_test += m.predict_proba(X_lgb_test)[:, 1] / 5
        all_test += seed_test
    lgb_test[horizon] = all_test / N_LGB_SEEDS
    print(f'LGB {horizon}h IPCW CV-bagged done')

LGB 24h IPCW CV-bagged done
LGB 48h IPCW CV-bagged done


In [7]:
W24 = 0.95
W48 = 0.45

test_blend = test_gbsa.copy()
test_blend[:, 1] = W24 * test_gbsa[:, 1] + (1 - W24) * lgb_test[24]
test_blend[:, 2] = W48 * test_gbsa[:, 2] + (1 - W48) * lgb_test[48]
test_blend[:, 3] = sigmoid_pred(dist_test, 5450, 50)

# Hard 5km cutoff
far_mask = dist_test >= 5000
test_blend[far_mask, :] = 0.0

test_final = enforce_monotonicity(test_blend)

submission = pd.DataFrame({
    'event_id': test_df['event_id'],
    'prob_12h': test_final[:, 0],
    'prob_24h': test_final[:, 1],
    'prob_48h': test_final[:, 2],
    'prob_72h': test_final[:, 3],
})

output_path = '/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
submission.describe().round(4)

Saved: /kaggle/working/submission.csv


,event_id,prob_12h,prob_24h,prob_48h,prob_72h
count,9.500000e+01,95.0000,95.0000,95.0000,95.0000
mean,5.695393e+07,0.1866,0.2460,0.2754,0.2947
std,2.329721e+07,0.3069,0.3870,0.4292,0.4583
min,1.066260e+07,0.0000,0.0000,0.0000,0.0000
25%,4.027536e+07,0.0000,0.0000,0.0000,0.0000
50%,5.480111e+07,0.0000,0.0000,0.0000,0.0000
75%,7.536942e+07,0.4458,0.7211,0.8846,1.0000
max,9.964946e+07,0.9390,0.9632,0.9839,1.0000
